# Case illustration: per-tooth outputs on a DENTEX panoramic radiograph

This notebook produces the illustrative case outputs of Fig. 7. For a selected DENTEX radiograph it runs the reference probe, the clipped residual update and the Mahalanobis gate on every annotated tooth (boxes from the DENTEX annotations), draws the results on the radiograph, and saves an HTML report, a PDF report and the annotated PNG.

If the embedding cache is missing, Section 3 builds it. Section 4b refits the probe, temperature, residual head and gate on DINOv3-L embeddings so that the same radiograph can be shown for both encoders.

Tooth numbering is FDI. In DENTEX, `category_id_1` (quadrant: 0 upper right, 1 upper left, 2 lower left, 3 lower right) and `category_id_2` (position 0 to 7) are both zero-based, so FDI = 10 × (quadrant + 1) + (position + 1).

The output illustrates the method for research purposes and is not a diagnosis. Teeth refused by the gate are not assigned a finding.

## 1. Environment, paths, DENTEX annotations, FDI mapping

In [ ]:
import sys, subprocess
def pipq(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p], check=False)
# Packages are listed in requirements.txt. Uncomment to install them into the active kernel.
# pipq("torch","torchvision","scikit-learn","pillow","matplotlib","tqdm",
#      "huggingface_hub","open_clip_torch","timm","ml_collections","reportlab")
import torch, numpy as np, os, glob, json, math, datetime, html, base64
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
DEVICE="mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/path/to/DENTEX/training_data/quadrant-enumeration-disease"
assert os.path.isdir(DATA_DIR), f"Not found: {DATA_DIR}"

IMG_DIR=None
for root,_,files in os.walk(DATA_DIR):
    if os.path.basename(root).lower()=="xrays" and any(f.lower().endswith((".png",".jpg")) for f in files):
        IMG_DIR=root; break
if IMG_DIR is None:
    for root,_,files in os.walk(DATA_DIR):
        if any(s in root.lower() for s in ("_result","output","checkpoint")): continue
        if any(f.lower().endswith((".png",".jpg")) for f in files): IMG_DIR=root; break
ANN_PATH=None
for root,_,files in os.walk(DATA_DIR):
    for f in files:
        if f.endswith(".json"):
            try: j=json.load(open(os.path.join(root,f)))
            except Exception: continue
            if isinstance(j,dict) and "annotations" in j and any("category_id_3" in a for a in j.get("annotations",[])):
                ANN_PATH=os.path.join(root,f); break
    if ANN_PATH: break
OUTPUT_DIR=os.path.join(DATA_DIR,"_results_option1"); os.makedirs(OUTPUT_DIR,exist_ok=True)
CACHE_DIR=os.path.join(OUTPUT_DIR,"emb_cache"); os.makedirs(CACHE_DIR,exist_ok=True)

coco=json.load(open(ANN_PATH)); cats3=sorted(coco["categories_3"],key=lambda c:c["id"])
DX_NAMES=[c["name"] for c in cats3]; NUM_DX=len(DX_NAMES); id2idx={c["id"]:i for i,c in enumerate(cats3)}
img_file={im["id"]:im["file_name"] for im in coco["images"]}
file2id={os.path.basename(v).lower():k for k,v in img_file.items()}

def to_fdi(q, p):
    # DENTEX category_id_1 (quadrant) and category_id_2 (position) are both zero-based.
    # Quadrant 0 upper right, 1 upper left, 2 lower left, 3 lower right; FDI = 10*(q+1) + (p+1),
    # so third molars are 18, 28, 38 and 48.
    return (int(q) + 1) * 10 + (int(p) + 1)

print("device:",DEVICE,"| images:",IMG_DIR,"| classes:",DX_NAMES)


## 2. Frozen Dental ViT-L encoder, crop and embedding helpers

In [ ]:
if not os.path.isdir("ViT-pytorch"):
    subprocess.run(["git","clone","--depth","1","https://github.com/jeonsworld/ViT-pytorch"],check=True)
sys.path.insert(0,"ViT-pytorch")
from models.modeling import VisionTransformer, CONFIGS
from huggingface_hub import hf_hub_download
ck=hf_hub_download(repo_id="Bryceee/Teeth_Visual_Experts_Models",
    filename="Teeth_Visual_Experts_ViT_L_periapical_images_cls_7diseases.bin")
state=torch.load(ck,map_location="cpu")
if isinstance(state,dict) and "state_dict" in state: state=state["state_dict"]
pos=state["transformer.embeddings.position_embeddings"].shape[1]
DENTAL_IMG=int(round(math.sqrt(pos-1)))*16; nh=state["head.weight"].shape[0]
m=VisionTransformer(CONFIGS["ViT-L_16"],img_size=DENTAL_IMG,num_classes=nh,zero_head=False)
m.load_state_dict(state,strict=False); m=m.to(DEVICE).eval()
import torchvision.transforms as T
dental_preprocess=T.Compose([T.Resize((DENTAL_IMG,DENTAL_IMG)),T.ToTensor(),T.Normalize([0.5]*3,[0.5]*3)])
class DE(torch.nn.Module):
    def __init__(s,v): super().__init__(); s.v=v
    @torch.no_grad()
    def forward(s,x): seq,_=s.v.transformer(x); return seq[:,0]
dental_model=DE(m).to(DEVICE).eval()

_img_cache={}
def load_image(path):
    if path not in _img_cache: _img_cache[path]=Image.open(path).convert("RGB")
    return _img_cache[path]
def crop_box(img,box,pad=0.15):
    W,H=img.size; x0,y0,x1,y1=box; w,h=x1-x0,y1-y0; px,py=w*pad,h*pad
    return img.crop((max(0,int(x0-px)),max(0,int(y0-py)),min(W,int(x1+px)),min(H,int(y1+py))))
@torch.no_grad()
def embed_crop(pil):
    x=dental_preprocess(pil).unsqueeze(0).to(DEVICE)
    return torch.nn.functional.normalize(dental_model(x).float().cpu(),dim=1).numpy()[0]
print("dental ViT-L loaded @",DENTAL_IMG)


## 3. Embeddings of all annotated teeth (cached after the first run)

Annotations are grouped into teeth with merged boxes, and every tooth is embedded once with the frozen encoder and cached. The first run takes a few minutes on Apple silicon; later runs load the cache.

In [ ]:
# group ALL teeth across the dataset (image_id, quadrant, position) -> label + merged box + file
tooth={}
for a in coco["annotations"]:
    key=(a["image_id"],a.get("category_id_1",-1),a.get("category_id_2",-1))
    if key not in tooth:
        tooth[key]={"image_id":a["image_id"],"q":a.get("category_id_1",-1),"p":a.get("category_id_2",-1),
                    "file":img_file[a["image_id"]],"y":np.zeros(NUM_DX,np.float32),"bbox":list(a["bbox"])}
    tooth[key]["y"][id2idx[a["category_id_3"]]]=1
    bx,by,bw,bh=tooth[key]["bbox"]; ax,ay,aw,ah=a["bbox"]
    x0,y0=min(bx,ax),min(by,ay); x1,y1=max(bx+bw,ax+aw),max(by+bh,ay+ah)
    tooth[key]["bbox"]=[x0,y0,x1-x0,y1-y0]
teeth_all=list(tooth.values())
Y=np.stack([t["y"] for t in teeth_all]); IMG_OF=np.array([t["image_id"] for t in teeth_all])

emb_path=os.path.join(CACHE_DIR,"emb_dental_vitl.npy")
if os.path.exists(emb_path) and np.load(emb_path).shape[0]==len(teeth_all):
    EMB=np.load(emb_path); print("loaded cached embeddings:",EMB.shape)
else:
    print(f"extracting embeddings for {len(teeth_all)} teeth (one-time, ~minutes on MPS)...")
    feats=[]
    for t in tqdm(teeth_all):
        img=load_image(os.path.join(IMG_DIR, os.path.basename(t["file"])))
        x,y,w,h=t["bbox"]
        feats.append(embed_crop(crop_box(img,[x,y,x+w,y+h])))
    EMB=np.stack(feats).astype(np.float32); np.save(emb_path,EMB)
    _img_cache.clear()
    print("cached embeddings:",EMB.shape,"->",emb_path)


## 4. Reference probe, temperature, residual head and Mahalanobis gate

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
DELTA=1.5; THRESH=0.50

ids=sorted(set(IMG_OF.tolist())); rng=np.random.RandomState(42); rng.shuffle(ids)
n=len(ids); te=set(ids[:int(n*.15)]); va=set(ids[int(n*.15):int(n*.30)])
tr=[i for i,im in enumerate(IMG_OF) if im not in te and im not in va]
va_=[i for i,im in enumerate(IMG_OF) if im in va]
Xtr,Ytr=EMB[tr],Y[tr]; Xva,Yva=EMB[va_],Y[va_]

baseline_probes=[LogisticRegression(max_iter=1000).fit(Xtr,Ytr[:,c]) if len(np.unique(Ytr[:,c]))>1 else None
                 for c in range(NUM_DX)]
def blogits(pr,X):
    o=np.zeros((len(X),NUM_DX),np.float32)
    for c,p in enumerate(pr): o[:,c]=p.decision_function(X) if p is not None else 0.0
    return o
Tp=nn.Parameter(torch.ones(1)); z=torch.tensor(blogits(baseline_probes,Xva)); yt=torch.tensor(Yva)
opt=torch.optim.LBFGS([Tp],lr=0.1,max_iter=60); bce=nn.BCEWithLogitsLoss()
opt.step(lambda:(opt.zero_grad(),bce(z/Tp.clamp(min=0.05),yt).backward(),bce(z/Tp.clamp(min=0.05),yt))[-1])
baseline_T=float(Tp.detach().clamp(min=0.05))
class EnvelopeHead(nn.Module):
    def __init__(s,d,k,delta): super().__init__(); s.delta=delta; s.net=nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Dropout(0.2),nn.Linear(256,k))
    def forward(s,x,b): return b+torch.clamp(s.net(x),-s.delta,s.delta)
envelope_head=EnvelopeHead(Xtr.shape[1],NUM_DX,DELTA).to(DEVICE)
btr=torch.tensor(blogits(baseline_probes,Xtr)/baseline_T)
dl=DataLoader(TensorDataset(torch.tensor(Xtr),btr,torch.tensor(Ytr)),batch_size=64,shuffle=True)
op=torch.optim.AdamW(envelope_head.parameters(),lr=1e-3,weight_decay=1e-4); cr=nn.BCEWithLogitsLoss()
for _ in range(60):
    envelope_head.train()
    for xb,bb,yb in dl:
        xb,bb,yb=xb.to(DEVICE),bb.to(DEVICE),yb.to(DEVICE); op.zero_grad(); cr(envelope_head(xb,bb),yb).backward(); op.step()
envelope_head.eval()
ood_mu=Xtr.mean(0); cov=np.cov(Xtr.T)+np.eye(Xtr.shape[1])*1e-2; ood_ci=np.linalg.inv(cov)
ood_threshold=float(np.quantile(np.einsum("ni,ij,nj->n",Xva-ood_mu,ood_ci,Xva-ood_mu),0.95))
print(f"envelope ready | T={baseline_T:.3f} | OOD threshold={ood_threshold:.1f}")


## 4b. Optional: DINOv3-L encoder

Run Sections 1 to 4 with Dental ViT-L first. This section replaces the embedding function with DINOv3-L, rebuilds the embeddings (cached separately) and refits the probe, temperature, residual head and gate on the DINOv3-L features. Then run Section 6 to render the same radiograph with DINOv3-L.

In [ ]:
# ============================================================================
# Optional DINOv3-L encoder (run after Sections 1 to 4).
# Replaces the embedding function with DINOv3-L, rebuilds the embeddings and refits
# the probe, temperature, residual head and gate on the DINOv3-L features.
# ============================================================================
import torch, numpy as np, os, torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from transformers import AutoModel
from tqdm.auto import tqdm

DINO_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
_dino = AutoModel.from_pretrained(DINO_ID).to(DEVICE).eval()
_dino_tf = T.Compose([
    T.Resize((224,224), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# --- override the embedder used everywhere downstream ---
@torch.no_grad()
def embed_crop(pil):                     # same name -> run_envelope() picks it up unchanged
    x=_dino_tf(pil.convert("RGB")).unsqueeze(0).to(DEVICE)
    o=_dino(pixel_values=x)
    f=getattr(o,"pooler_output",None)
    if f is None: f=o.last_hidden_state[:,0]
    return torch.nn.functional.normalize(f.float().cpu(),dim=1).numpy()[0]
print("embedder -> DINOv3 (224px)")

# --- rebuild all-teeth embeddings with DINOv3 (cached separately) ---
emb_path_dino=os.path.join(CACHE_DIR,"emb_dino_vitl.npy")
if os.path.exists(emb_path_dino) and np.load(emb_path_dino).shape[0]==len(teeth_all):
    EMB=np.load(emb_path_dino); print("loaded cached DINOv3 embeddings:",EMB.shape)
else:
    print(f"extracting DINOv3 embeddings for {len(teeth_all)} teeth (one-time)...")
    feats=[]
    for t in tqdm(teeth_all):
        img=load_image(os.path.join(IMG_DIR, os.path.basename(t["file"])))
        x,y,w,h=t["bbox"]
        feats.append(embed_crop(crop_box(img,[x,y,x+w,y+h])))
    EMB=np.stack(feats).astype(np.float32); np.save(emb_path_dino,EMB)
    _img_cache.clear()
    print("cached DINOv3 embeddings:",EMB.shape,"->",emb_path_dino)

# --- refit envelope on DINOv3 features (identical recipe to cell 8) ---
Xtr,Ytr=EMB[tr],Y[tr]; Xva,Yva=EMB[va_],Y[va_]
baseline_probes=[LogisticRegression(max_iter=1000).fit(Xtr,Ytr[:,c]) if len(np.unique(Ytr[:,c]))>1 else None
                 for c in range(NUM_DX)]
Tp=nn.Parameter(torch.ones(1)); z=torch.tensor(blogits(baseline_probes,Xva)); yt=torch.tensor(Yva)
opt=torch.optim.LBFGS([Tp],lr=0.1,max_iter=60); bce=nn.BCEWithLogitsLoss()
opt.step(lambda:(opt.zero_grad(),bce(z/Tp.clamp(min=0.05),yt).backward(),bce(z/Tp.clamp(min=0.05),yt))[-1])
baseline_T=float(Tp.detach().clamp(min=0.05))
envelope_head=EnvelopeHead(Xtr.shape[1],NUM_DX,DELTA).to(DEVICE)
btr=torch.tensor(blogits(baseline_probes,Xtr)/baseline_T)
dl=DataLoader(TensorDataset(torch.tensor(Xtr),btr,torch.tensor(Ytr)),batch_size=64,shuffle=True)
op=torch.optim.AdamW(envelope_head.parameters(),lr=1e-3,weight_decay=1e-4); cr=nn.BCEWithLogitsLoss()
for _ in range(60):
    envelope_head.train()
    for xb,bb,yb in dl:
        xb,bb,yb=xb.to(DEVICE),bb.to(DEVICE),yb.to(DEVICE); op.zero_grad(); cr(envelope_head(xb,bb),yb).backward(); op.step()
envelope_head.eval()
ood_mu=Xtr.mean(0); cov=np.cov(Xtr.T)+np.eye(Xtr.shape[1])*1e-2; ood_ci=np.linalg.inv(cov)
ood_threshold=float(np.quantile(np.einsum("ni,ij,nj->n",Xva-ood_mu,ood_ci,Xva-ood_mu),0.95))
print(f"DINOv3 envelope ready | T={baseline_T:.3f} | OOD threshold={ood_threshold:.1f}")
print("Run Section 6 to render the selected radiograph with DINOv3-L.")


## 5. Helpers: file selection, overlay, inference, HTML and PDF reports

In [ ]:
# Radiograph and output folder. Set MANUAL_IMAGE and MANUAL_SAVE_DIR to skip the Tk dialogs.
MANUAL_IMAGE    = "/path/to/DENTEX/training_data/quadrant-enumeration-disease/xrays/train_564.png"
MANUAL_SAVE_DIR = "./case_reports"
# (set either to None to use the file dialog instead)

def _run_dialog(kind, **kw):
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()                      # standard, supported way to hide the root
    root.attributes("-topmost", True)    # bring the dialog above Jupyter
    root.update()                        # let the event loop settle before opening
    try:
        if kind == "open":
            path = filedialog.askopenfilename(parent=root, **kw)
        else:
            path = filedialog.askdirectory(parent=root, **kw)
    finally:
        root.update()                    # flush pending events -> no ghost window
        root.destroy()
    return path or None

def pick_image_file(title="Select a DENTEX panoramic image", initialdir=None):
    if MANUAL_IMAGE:
        print("using MANUAL_IMAGE:", MANUAL_IMAGE); return MANUAL_IMAGE
    return _run_dialog("open", title=title, initialdir=initialdir,
        filetypes=[("Images","*.png *.jpg *.jpeg *.PNG *.JPG"),("All files","*.*")])

def pick_save_folder(title="Choose a folder to save the report"):
    if MANUAL_SAVE_DIR:
        import os; os.makedirs(MANUAL_SAVE_DIR, exist_ok=True)
        print("using MANUAL_SAVE_DIR:", MANUAL_SAVE_DIR); return MANUAL_SAVE_DIR
    return _run_dialog("dir", title=title)


In [ ]:
# ---- native PDF builder (reportlab; reliable on macOS, no external binaries) ----
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib import colors as _rc
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
                                Table, TableStyle)
from PIL import Image as _PILImage

def build_pdf_report(path, meta, anno_png, table_rows, narrative_lines):
    NAVY=_rc.HexColor("#1a3a5c"); GREY=_rc.HexColor("#5b6770"); LIGHT=_rc.HexColor("#f4f6f8")
    S=getSampleStyleSheet()
    h1=ParagraphStyle("h1",parent=S["Heading1"],textColor=NAVY,fontSize=16,spaceAfter=2)
    sub=ParagraphStyle("sub",parent=S["Normal"],textColor=GREY,fontSize=8.5,spaceAfter=8)
    disc=ParagraphStyle("disc",parent=S["Normal"],fontSize=8.5,textColor=_rc.HexColor("#444"),
                        backColor=LIGHT,borderPadding=6,spaceAfter=10)
    h2=ParagraphStyle("h2",parent=S["Heading2"],textColor=NAVY,fontSize=11,spaceBefore=10,spaceAfter=4)
    body=ParagraphStyle("body",parent=S["Normal"],fontSize=9.5,leading=13,spaceAfter=4)
    cellp=ParagraphStyle("cellp",parent=S["Normal"],fontSize=8,leading=10)
    foot=ParagraphStyle("foot",parent=S["Normal"],fontSize=7.5,textColor=GREY,spaceBefore=10)
    doc=SimpleDocTemplate(path,pagesize=A4,topMargin=15*mm,bottomMargin=15*mm,
                          leftMargin=15*mm,rightMargin=15*mm,title="Findings Report")
    page_w=A4[0]-30*mm
    E=[Paragraph("Panoramic Radiograph &mdash; Findings Report",h1),
       Paragraph(f"Image: {meta['image']} &nbsp;|&nbsp; {meta['date']} &nbsp;|&nbsp; "
                 f"dental ViT-L envelope, &delta;={meta['delta']} &nbsp;|&nbsp; threshold {meta['thr']}",sub),
       Paragraph("<b>Decision support only &mdash; not a diagnosis or treatment plan.</b> "
                 "Calibrated, bounded model with an out-of-distribution gate; "
                 f"{meta['n_ref']} tooth/teeth deferred. A panoramic alone is insufficient for treatment "
                 "decisions (requires clinical exam, vitality testing, probing, history, symptoms).",disc),
       Paragraph("Annotated panoramic",h2)]
    iw,ih=_PILImage.open(anno_png).size
    E.append(RLImage(anno_png,width=page_w,height=page_w*ih/iw))
    E.append(Spacer(1,6))
    E.append(Paragraph("Per-tooth findings (FDI numbering)",h2))
    data=[["Tooth","Radiographic finding (calibrated confidence)","Clinical confirmation required"]]
    for r in table_rows:
        data.append([Paragraph(r[0],cellp),Paragraph(r[1],cellp),Paragraph(r[2],cellp)])
    t=Table(data,colWidths=[15*mm,72*mm,page_w-15*mm-72*mm])
    t.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,0),NAVY),("TEXTCOLOR",(0,0),(-1,0),_rc.white),
        ("FONTSIZE",(0,0),(-1,0),8),("FONTNAME",(0,0),(-1,0),"Helvetica-Bold"),
        ("VALIGN",(0,0),(-1,-1),"TOP"),("LINEBELOW",(0,0),(-1,-1),0.4,_rc.HexColor("#e3e7ea")),
        ("TOPPADDING",(0,0),(-1,-1),4),("BOTTOMPADDING",(0,0),(-1,-1),4),
        ("LEFTPADDING",(0,0),(-1,-1),5),("RIGHTPADDING",(0,0),(-1,-1),5)]))
    E.append(t)
    E.append(Paragraph("Narrative summary",h2))
    for l in narrative_lines: E.append(Paragraph(l,body))
    E.append(Paragraph("Confidences are calibrated probabilities; each finding lies within the model's "
                       "&plusmn;&delta; bound. OOD teeth are excluded from findings by design. No treatment "
                       "is recommended or implied.",foot))
    doc.build(E)
    return path


In [ ]:
# DENTEX-house-style palette: box colors + brighter text colors for dark label bars
COND_COLOR={"Impacted":(150,120,200),"Caries":(230,190,60),"Periapical Lesion":(70,140,210),"Deep Caries":(220,80,80)}
COND_TEXT ={"Impacted":(190,165,235),"Caries":(245,210,90),"Periapical Lesion":(120,185,240),"Deep Caries":(245,130,130)}
REFUSE_COLOR=(150,160,170); NOFIND_COLOR=(90,180,110)
CONFIRM={"Impacted":"clinical examination; CBCT to assess root morphology and nerve proximity",
 "Caries":"direct visual-tactile examination; bitewing for proximal extent",
 "Periapical Lesion":"pulp vitality testing (cold/electric); percussion and palpation",
 "Deep Caries":"clinical cavity-depth assessment; pulp vitality testing for pulpal involvement"}
TREATMENT_BANNED=["extract","extraction","root canal","rct","crown","filling","prescribe","restore","therapy","treatment plan"]
def _font(sz):
    for p in ["/System/Library/Fonts/Supplemental/Arial Bold.ttf","/Library/Fonts/Arial Bold.ttf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"]:
        try: return ImageFont.truetype(p,sz)
        except Exception: pass
    return ImageFont.load_default()
def teeth_for_image(image_id):
    tt={}
    for a in coco["annotations"]:
        if a["image_id"]!=image_id: continue
        key=(a.get("category_id_1",-1),a.get("category_id_2",-1))
        if key not in tt: tt[key]={"fdi":to_fdi(*key),"bbox":list(a["bbox"])}
        bx,by,bw,bh=tt[key]["bbox"]; ax,ay,aw,ah=a["bbox"]
        x0,y0=min(bx,ax),min(by,ay); x1,y1=max(bx+bw,ax+aw),max(by+bh,ay+ah)
        tt[key]["bbox"]=[x0,y0,x1-x0,y1-y0]
    out=[]
    for t in tt.values():
        x,y,w,h=t["bbox"]; out.append({"fdi":t["fdi"],"bbox":[x,y,x+w,y+h]})
    return sorted(out,key=lambda r:r["fdi"])
def run_envelope(records, pano_img):
    for r in records:
        emb=embed_crop(crop_box(pano_img,r["bbox"]))
        bl=blogits(baseline_probes,emb[None,:])/baseline_T
        with torch.no_grad():
            fin=envelope_head(torch.tensor(emb[None,:]).to(DEVICE),torch.tensor(bl).to(DEVICE)).cpu().numpy()[0]
        r["probs"]=[float(1/(1+np.exp(-v))) for v in fin]
        mdv=float(np.einsum("i,ij,j->",emb-ood_mu,ood_ci,emb-ood_mu)); r["ood"]=mdv; r["accept"]=bool(mdv<=ood_threshold)
    return records
def primary(r):
    if not r["accept"]: return("refused",None)
    bi,bp=-1,0.0
    for i,p in enumerate(r["probs"]):
        if p>=THRESH and p>bp: bi,bp=i,p
    return (DX_NAMES[bi],bp) if bi>=0 else ("nofinding",None)
def _shrink(box, frac=0.12):
    x0,y0,x1,y1=box; w,h=x1-x0,y1-y0; dx,dy=w*frac/2.0,h*frac/2.0
    return [x0+dx,y0+dy,x1-dx,y1-dy]
def annotate(img,recs,shrink_frac=0.12):
    # DENTEX-house style: clean colored boxes, dark label bars with full diagnosis, colored text.
    im=img.convert("RGB").copy(); d=ImageDraw.Draw(im,"RGBA"); W,H=im.size
    lw=max(2,W//500); f=_font(max(16,W//85))
    recs_sorted=sorted(recs,key=lambda r:r["bbox"][0]); placed=[]
    for r in recs_sorted:
        x0,y0,x1,y1=[int(v) for v in _shrink(r["bbox"],shrink_frac)]; c,cf=primary(r)
        if c=="refused": box=REFUSE_COLOR; txt=(235,235,235); lab=f'{r["fdi"]}  Deferred (OOD)'
        elif c=="nofinding": box=NOFIND_COLOR; txt=(210,240,215); lab=f'{r["fdi"]}'
        else: box=COND_COLOR[c]; txt=COND_TEXT[c]; lab=f'{r["fdi"]}  {c}  {cf:.0%}'
        d.rectangle([x0,y0,x1,y1],outline=box,width=lw)
        tb=d.textbbox((0,0),lab,font=f); tw,th=tb[2]-tb[0],tb[3]-tb[1]
        ly=y0-th-8
        collide=any((not(x0+tw<px0 or x0>px1)) and abs(ly-ply0)<th+6 for (px0,px1,ply0,ply1) in placed[-4:])
        if ly<2 or collide: ly=y1+4
        d.rectangle([x0,ly,x0+tw+12,ly+th+8], fill=(0,0,0,205))
        d.text((x0+6,ly+3),lab,fill=txt,font=f)
        placed.append((x0,x0+tw+12,ly,ly+th+8))
    items=[(n,COND_COLOR[n],COND_TEXT[n]) for n in DX_NAMES]+[("Refused (OOD)",REFUSE_COLOR,(235,235,235)),("No finding",NOFIND_COLOR,(210,240,215))]
    f2=_font(max(15,W//95)); pad=12; lh=max(24,W//58); bw=320; bh=pad*2+lh*len(items); lx,ly0=W-bw-16,16
    d.rectangle([lx,ly0,lx+bw,ly0+bh],fill=(0,0,0,205))
    for i,(nm,bc,tc) in enumerate(items):
        yy=ly0+pad+i*lh; d.rectangle([lx+pad,yy+3,lx+pad+22,yy+25],outline=bc,width=3); d.text((lx+pad+34,yy+3),nm,fill=tc,font=f2)
    return im
def select_findings(t):
    if not t["accept"]: return {"fdi":t["fdi"],"status":"refused","findings":[],"ood":t.get("ood")}
    fs=[{"condition":DX_NAMES[i],"confidence":round(float(p),2)} for i,p in enumerate(t["probs"]) if p>=THRESH]
    return {"fdi":t["fdi"],"status":"assessed","findings":fs,"ood":t.get("ood")}
def narrative(sel):
    wf=[s for s in sel if s["status"]=="assessed" and s["findings"]]; ref=[s for s in sel if s["status"]=="refused"]
    L=[]
    for s in wf:
        parts=[f'{f["condition"].lower()} (calibrated confidence {f["confidence"]:.0%}, within the \u00b1\u03b4 bound)' for f in s["findings"]]
        L.append(f'Tooth {s["fdi"]}: radiographic findings consistent with '+"; ".join(parts)+".")
    if not wf: L.append("No condition exceeded the calibrated reporting threshold on the assessed teeth.")
    if ref: L.append("Deferred for direct clinician review (out-of-distribution): "+", ".join(str(s["fdi"]) for s in ref)+".")
    need=sorted({CONFIRM[f["condition"]] for s in wf for f in s["findings"]})
    if need: L.append("Before any treatment decision, confirm clinically: "+"; ".join(need)+".")
    txt=" ".join(L).lower(); assert not [w for w in TREATMENT_BANNED if w in txt], "guardrail"
    L.append("This findings summary is decision support only; it is not a diagnosis or a treatment plan.")
    return L
def table_rows_for_pdf(sel):
    rows=[]
    for s in sorted(sel,key=lambda z:z["fdi"]):
        if s["status"]=="refused": rows.append([str(s["fdi"]),f"Not assessed &mdash; OOD (Mahalanobis {s['ood']:.0f}); deferred",""])
        elif not s["findings"]: rows.append([str(s["fdi"]),"No reportable finding above threshold",""])
        else:
            fr="; ".join(f'{f["condition"]} {f["confidence"]:.0%}' for f in s["findings"])
            cf="; ".join(sorted({CONFIRM[f["condition"]] for f in s["findings"]}))
            rows.append([str(s["fdi"]),fr,cf])
    return rows
def build_html(sel,narr,meta,anno_path):
    def badge(p):
        c="#3a7d44" if p>=0.75 else ("#c9a227" if p>=0.6 else "#5b6770")
        return f"<span style='background:{c};color:#fff;padding:1px 7px;border-radius:10px;font-size:12px'>{p:.0%}</span>"
    b=base64.b64encode(open(anno_path,"rb").read()).decode()
    rows=[]
    for s in sorted(sel,key=lambda z:z["fdi"]):
        if s["status"]=="refused":
            rows.append(f"<tr style='background:#fbeaea'><td style='font-weight:700;white-space:nowrap'>{s['fdi']}</td><td colspan='2' style='color:#a83232'>Not assessed \u2014 OOD (Mahalanobis {s['ood']:.0f}); deferred</td></tr>")
        elif not s["findings"]:
            rows.append(f"<tr><td style='font-weight:700;white-space:nowrap'>{s['fdi']}</td><td style='color:#5b6770'>No reportable finding above threshold</td><td></td></tr>")
        else:
            items="<br>".join(html.escape(f["condition"])+" &nbsp;"+badge(f["confidence"]) for f in s["findings"])
            confs="; ".join(sorted({CONFIRM[f["condition"]] for f in s["findings"]}))
            rows.append(f"<tr><td style='font-weight:700;white-space:nowrap'>{s['fdi']}</td><td>{items}</td><td style='font-size:12px;color:#5b6770'>{confs}</td></tr>")
    nh="".join("<p style='margin:6px 0'>"+html.escape(l)+"</p>" for l in narr)
    nf=sum(len(s["findings"]) for s in sel); nr=sum(s["status"]=="refused" for s in sel)
    CSS="<style>body{font-family:Georgia,serif;color:#222;max-width:900px;margin:24px auto;line-height:1.5}h1{color:#1a3a5c;font-size:22px;margin:0 0 2px}.sub{color:#5b6770;font-size:13px}.disc{background:#f4f6f8;border-left:4px solid #c9a227;padding:10px 14px;font-size:13px;color:#444;margin:12px 0}table{border-collapse:collapse;width:100%;margin:10px 0}th{background:#1a3a5c;color:#fff;text-align:left;padding:7px 10px;font-size:13px}td{border-bottom:1px solid #e3e7ea;padding:7px 10px;font-size:14px;vertical-align:top}h2{color:#1a3a5c;font-size:15px;margin:18px 0 4px;border-bottom:1px solid #e3e7ea;padding-bottom:3px}.foot{color:#5b6770;font-size:11px;margin-top:18px;border-top:1px solid #e3e7ea;padding-top:8px}</style>"
    return ("<!doctype html><html><head><meta charset='utf-8'>"+CSS+"</head><body>"
      f"<h1>Panoramic Radiograph \u2014 Findings Report</h1>"
      f"<p class='sub'>Image: {html.escape(meta['image'])} &nbsp;|&nbsp; {meta['date']} &nbsp;|&nbsp; dental ViT-L envelope, &delta;={meta['delta']} &nbsp;|&nbsp; threshold {meta['thr']}</p>"
      f"<div class='disc'><b>Decision support only \u2014 not a diagnosis or treatment plan.</b> Calibrated, bounded model with an out-of-distribution gate; {nr} tooth/teeth deferred. A panoramic alone is insufficient for treatment decisions (requires clinical exam, vitality testing, probing, history, symptoms).</div>"
      f"<h2>Annotated panoramic</h2><img style='width:100%;border:1px solid #ccc;border-radius:4px' src='data:image/png;base64,{b}'/>"
      f"<h2>Per-tooth findings <span style='font-weight:400;font-size:12px;color:#5b6770'>({nf} finding(s); FDI numbering)</span></h2>"
      f"<table><tr><th style='width:64px'>Tooth</th><th>Radiographic finding (calibrated confidence)</th><th style='width:36%'>Clinical confirmation required before treatment</th></tr>{''.join(rows)}</table>"
      f"<h2>Narrative summary</h2>{nh}"
      f"<p class='foot'>Confidences are calibrated probabilities; each finding lies within the model's \u00b1\u03b4 bound. OOD teeth are excluded from findings by design. No treatment is recommended or implied.</p></body></html>")
print("helpers ready")


## 6. Run: select a radiograph, render and save

The cell takes the radiograph from `MANUAL_IMAGE` (or a file dialog when it is `None`), runs the method on its teeth, shows the overlay and saves the HTML report, PDF report and annotated PNG to `MANUAL_SAVE_DIR` (or a folder chosen in a dialog). On macOS a Tk dialog can open behind the browser window.

In [ ]:
# 1) pick image
img_path = pick_image_file(initialdir=IMG_DIR)
assert img_path, "No image selected."
base = os.path.basename(img_path).lower()
assert base in file2id, (f"'{os.path.basename(img_path)}' is not in the DENTEX annotations (no tooth "
                         "boxes available). Choose a DENTEX image from the xrays folder.")
image_id = file2id[base]
records = teeth_for_image(image_id)
pano_img = Image.open(img_path).convert("RGB")
print(f"selected {os.path.basename(img_path)} | {len(records)} teeth | running envelope...")

# 2) run envelope live + overlay
run_envelope(records, pano_img)
annotated = annotate(pano_img, records)
anno_tmp = "_annotated_tmp.png"; annotated.save(anno_tmp)
from IPython.display import display, HTML
display(annotated)

# 3) report objects
sel = [select_findings(t) for t in records]
narr = narrative(sel)
meta = {"image":os.path.basename(img_path),"date":datetime.date.today().isoformat(),
        "delta":DELTA,"thr":f"{int(THRESH*100)}%","n_ref":sum(s['status']=='refused' for s in sel)}
report_html = build_html(sel, narr, meta, anno_tmp)
display(HTML(report_html))

# 4) pick folder + save HTML, PDF, PNG
folder = pick_save_folder()
assert folder, "No save folder chosen."
stem = os.path.splitext(os.path.basename(img_path))[0]
html_out = os.path.join(folder, f"findings_{stem}.html")
pdf_out  = os.path.join(folder, f"findings_{stem}.pdf")
png_out  = os.path.join(folder, f"annotated_{stem}.png")
annotated.save(png_out)
open(html_out,"w").write(report_html)
build_pdf_report(pdf_out, meta, anno_tmp, table_rows_for_pdf(sel), narr)
print("\nSaved:\n ", html_out, "\n ", pdf_out, "\n ", png_out)
